<a href="https://colab.research.google.com/github/mejian1/ExopherGeneExpressionProfiling/blob/main/RNAseq_DGE_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RNA-seq Differential Gene Expression Analysis

This notebook performs a complete differential gene expression (DGE) analysis pipeline
for RNA-seq data. The workflow covers:

1. **Install dependencies** – PyDESeq2 and supporting libraries
2. **Load & inspect raw count data** – from TSV/CSV files or simulated data
3. **Quality control** – library size distribution, PCA, sample correlation
4. **Normalization** – size-factor estimation (DESeq2 median-of-ratios)
5. **Differential expression** – Wald test via PyDESeq2
6. **Results filtering** – apply adjusted p-value and fold-change thresholds
7. **Visualization** – Volcano plot, MA plot, heatmap of top DEGs, PCA
8. **Export results** – write DEG tables to TSV files

**Dataset context**: *C. elegans* sbp-1 RNAi knockdown RNA-seq experiment
(GEO accession GSE70692). Upload the following files before running:
- `GSE70692_sbp1RNAi_raw_counts.tsv` – integer read-count matrix
  (genes × samples, with a `sample_info.tsv` metadata file)

If raw count files are not available the notebook generates a synthetic
dataset that mirrors the structure of the experiment so all code cells
can still be executed end-to-end.


## 1. Install Dependencies


In [ ]:
# Install PyDESeq2 and additional plotting / analysis libraries
!pip install pydeseq2 adjustText --quiet
!pip install matplotlib seaborn scikit-learn scipy statsmodels --quiet

## 2. Load and Inspect Count Data


In [ ]:
import os
import numpy as np
import pandas as pd

COUNTS_FILE  = '/content/GSE70692_sbp1RNAi_raw_counts.tsv'
METADATA_FILE = '/content/sample_info.tsv'

# ---------------------------------------------------------------
# If real data files are not uploaded, create a synthetic dataset
# that mirrors the sbp-1 RNAi experiment structure:
#   3 control samples vs 3 sbp-1 RNAi samples, ~20 000 genes
# ---------------------------------------------------------------
if not os.path.exists(COUNTS_FILE):
    print("[INFO] Count file not found – generating synthetic dataset.")
    rng = np.random.default_rng(42)
    n_genes, n_samples = 20_000, 6
    sample_ids  = [f"ctrl_{i}" for i in range(1, 4)] + [f"sbp1_{i}" for i in range(1, 4)]
    gene_ids    = [f"WBGene{str(i).zfill(8)}" for i in range(1, n_genes + 1)]

    # Base counts drawn from negative binomial
    base_mu = rng.gamma(shape=1, scale=100, size=n_genes)
    counts  = rng.negative_binomial(n=10, p=10 / (10 + base_mu[:, None]),
                                    size=(n_genes, n_samples))

    # Introduce ~1 000 truly differentially expressed genes
    n_de = 1_000
    de_idx = rng.choice(n_genes, n_de, replace=False)
    fold_changes = rng.choice([-1, 1], n_de) * rng.uniform(1, 4, n_de)
    counts[de_idx, 3:] = np.clip(
        (counts[de_idx, :3] * (2 ** fold_changes[:, None])).astype(int), 0, None
    )

    counts_df = pd.DataFrame(counts, index=gene_ids, columns=sample_ids)
    counts_df.to_csv(COUNTS_FILE, sep='\t')

    metadata = pd.DataFrame({'condition': ['control'] * 3 + ['sbp1_RNAi'] * 3},
                            index=sample_ids)
    metadata.to_csv(METADATA_FILE, sep='\t')
    print(f"[INFO] Synthetic counts: {counts_df.shape[0]} genes × {counts_df.shape[1]} samples")

# Load count matrix (genes as rows, samples as columns)
counts_df = pd.read_csv(COUNTS_FILE, sep='\t', index_col=0)
metadata  = pd.read_csv(METADATA_FILE, sep='\t', index_col=0)

print("Count matrix shape:", counts_df.shape)
print("\nSample metadata:")
display(metadata)
print("\nFirst 5 genes:")
display(counts_df.head())

## 3. Quality Control

Inspect library sizes, per-gene count distributions, and sample-to-sample
correlations before normalization.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

conditions = metadata['condition']
palette    = {'control': '#1B9E77', 'sbp1_RNAi': '#D95F02'}
colors     = [palette[c] for c in conditions]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Library sizes ---
lib_sizes = counts_df.sum(axis=0) / 1e6
axes[0].bar(range(len(lib_sizes)), lib_sizes, color=colors)
axes[0].set_xticks(range(len(lib_sizes)))
axes[0].set_xticklabels(lib_sizes.index, rotation=45, ha='right')
axes[0].set_xlabel('Sample')
axes[0].set_ylabel('Library size (M reads)')
axes[0].set_title('Library Sizes')
from matplotlib.patches import Patch
axes[0].legend(handles=[Patch(color=v, label=k) for k, v in palette.items()])

# --- Log2 count distribution (pseudo-count + 1) ---
log_counts = np.log2(counts_df + 1)
axes[1].boxplot(log_counts.values, labels=log_counts.columns,
                patch_artist=True,
                boxprops=dict(facecolor='lightblue'))
axes[1].set_xlabel('Sample')
axes[1].set_ylabel('log2(count + 1)')
axes[1].set_title('Count Distribution per Sample')
plt.setp(axes[1].get_xticklabels(), rotation=45, ha='right')

# --- Sample correlation heatmap ---
corr_mat = log_counts.corr()
sns.heatmap(corr_mat, ax=axes[2], annot=True, fmt='.3f',
            cmap='coolwarm', vmin=0.8, vmax=1.0,
            linewidths=0.5)
axes[2].set_title('Sample Correlation (log2 counts)')

plt.tight_layout()
plt.show()
print("Minimum library size:", lib_sizes.min().round(2), "M reads")
print("Maximum library size:", lib_sizes.max().round(2), "M reads")

## 4 & 5. Normalization and Differential Expression with PyDESeq2

PyDESeq2 implements the DESeq2 method (Love et al. 2014) in Python.
It estimates size factors using the median-of-ratios method and fits
a negative binomial GLM per gene, followed by a Wald test.


In [ ]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds  import DeseqStats

# PyDESeq2 expects samples as rows and genes as columns
counts_T = counts_df.T

# Remove genes with zero counts in all samples
counts_T = counts_T.loc[:, counts_T.sum(axis=0) > 0]
print(f"Genes after filtering zero-count genes: {counts_T.shape[1]}")

# Build DESeq2 dataset
dds = DeseqDataSet(
    counts    = counts_T,
    metadata  = metadata,
    design_factors = 'condition',
    refit_cooks    = True,
    n_cpus         = 4,
)

# Run DESeq2 (size-factor estimation + dispersion + GLM fitting)
dds.deseq2()

# Extract normalized counts for downstream visualization
norm_counts = dds.layers['normed_counts']
norm_counts_df = pd.DataFrame(norm_counts,
                               index  = counts_T.index,
                               columns= counts_T.columns)
print("\nSize factors:")
print(dds.size_factors)


In [ ]:
# Wald test: sbp1_RNAi vs control
stat_res = DeseqStats(
    dds,
    contrast = ('condition', 'sbp1_RNAi', 'control'),
    alpha    = 0.05,
    cooks_filter    = True,
    independent_filter = True,
)
stat_res.run_wald_test()
stat_res.p_value_adjustment()   # Benjamini-Hochberg FDR

# Collect results into a DataFrame
results = stat_res.results_df.copy()
results.index.name = 'gene_id'
results = results.sort_values('padj', na_position='last')

print(f"Total genes tested: {len(results)}")
print(f"Significant DEGs (padj < 0.05): {(results['padj'] < 0.05).sum()}")
display(results.head(10))

## 6. Filter and Summarize DEG Results


In [ ]:
PADJ_THRESHOLD = 0.05
LFC_THRESHOLD  = 1.0     # |log2 fold change| >= 1  (i.e., >=2-fold change)

sig = results.dropna(subset=['padj'])
sig = sig[sig['padj'] < PADJ_THRESHOLD]

deg_up   = sig[sig['log2FoldChange'] >  LFC_THRESHOLD]
deg_down = sig[sig['log2FoldChange'] < -LFC_THRESHOLD]
deg_all  = sig[sig['log2FoldChange'].abs() >= LFC_THRESHOLD]

print(f"Up-regulated DEGs   (padj<{PADJ_THRESHOLD}, logFC>{LFC_THRESHOLD}):  {len(deg_up)}")
print(f"Down-regulated DEGs (padj<{PADJ_THRESHOLD}, logFC<-{LFC_THRESHOLD}): {len(deg_down)}")
print(f"Total DEGs          (padj<{PADJ_THRESHOLD}, |logFC|>={LFC_THRESHOLD}): {len(deg_all)}")

print("\nTop up-regulated genes:")
display(deg_up.sort_values('log2FoldChange', ascending=False).head(10))

print("\nTop down-regulated genes:")
display(deg_down.sort_values('log2FoldChange').head(10))

## 7. Visualization

### 7a. Volcano Plot


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plot_df = results.dropna(subset=['padj', 'log2FoldChange']).copy()
# Clip p-values to avoid log10(0); 1e-100 is well within float64 range
plot_df['-log10_padj'] = -np.log10(plot_df['padj'].clip(lower=1e-100))

# Assign color categories
def assign_color(row):
    if row['padj'] < PADJ_THRESHOLD and row['log2FoldChange'] >= LFC_THRESHOLD:
        return 'Up-regulated'
    elif row['padj'] < PADJ_THRESHOLD and row['log2FoldChange'] <= -LFC_THRESHOLD:
        return 'Down-regulated'
    else:
        return 'Not significant'

plot_df['category'] = plot_df.apply(assign_color, axis=1)
color_map = {'Up-regulated': '#D95F02', 'Down-regulated': '#1B9E77', 'Not significant': '#AAAAAA'}

fig, ax = plt.subplots(figsize=(10, 7))
for cat, grp in plot_df.groupby('category'):
    ax.scatter(grp['log2FoldChange'], grp['-log10_padj'],
               c=color_map[cat], alpha=0.5, s=8, label=cat, rasterized=True)

# Threshold lines
ax.axhline(-np.log10(PADJ_THRESHOLD), color='black', linestyle='--', linewidth=0.8, alpha=0.7)
ax.axvline( LFC_THRESHOLD,            color='black', linestyle='--', linewidth=0.8, alpha=0.7)
ax.axvline(-LFC_THRESHOLD,            color='black', linestyle='--', linewidth=0.8, alpha=0.7)

# Label top 10 genes by -log10(padj)
top_genes = plot_df.nlargest(10, '-log10_padj')
for _, row in top_genes.iterrows():
    ax.annotate(row.name, (row['log2FoldChange'], row['-log10_padj']),
                fontsize=7, ha='center', va='bottom',
                xytext=(0, 4), textcoords='offset points')

ax.set_xlabel('log2 Fold Change (sbp-1 RNAi / control)', fontsize=12)
ax.set_ylabel('-log10(adjusted p-value)',                 fontsize=12)
ax.set_title('Volcano Plot – sbp-1 RNAi vs Control (RNA-seq)',  fontsize=13)
ax.legend(markerscale=2)
ax.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig('volcano_plot_sbp1_RNAi.png', dpi=150, bbox_inches='tight')
plt.show()
print("Volcano plot saved to volcano_plot_sbp1_RNAi.png")

### 7b. MA Plot (log fold change vs mean expression)


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

for cat, grp in plot_df.groupby('category'):
    ax.scatter(np.log2(grp['baseMean'] + 1), grp['log2FoldChange'],
               c=color_map[cat], alpha=0.4, s=6, label=cat, rasterized=True)

ax.axhline(0, color='darkblue', linewidth=1)
ax.set_xlabel('log2 Mean Expression (baseMean)', fontsize=12)
ax.set_ylabel('log2 Fold Change',                fontsize=12)
ax.set_title('MA Plot – sbp-1 RNAi vs Control',  fontsize=13)
ax.legend(markerscale=3)
ax.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig('MA_plot_sbp1_RNAi.png', dpi=150, bbox_inches='tight')
plt.show()
print("MA plot saved to MA_plot_sbp1_RNAi.png")

### 7c. Heatmap of Top 50 DEGs


In [ ]:
import seaborn as sns

top50 = deg_all.nsmallest(50, 'padj').index

# Z-score normalize across samples
heat_data = norm_counts_df.loc[:, top50].T
mean_expr = heat_data.mean(axis=1).values[:, None]
std_expr  = heat_data.std(axis=1).values[:, None]
heat_z    = (heat_data - mean_expr) / std_expr

# Column colour bar for conditions
col_colors = pd.Series(
    [palette[conditions[s]] for s in heat_z.columns],
    index=heat_z.columns
)

g = sns.clustermap(
    heat_z,
    col_colors   = col_colors,
    cmap         = 'vlag',
    center       = 0,
    method       = 'average',
    figsize      = (10, 14),
    dendrogram_ratio = (0.1, 0.2),
    cbar_pos     = (0, 0.2, 0.03, 0.4),
)
g.fig.suptitle('Heatmap – Top 50 DEGs (Z-score)', y=1.01, fontsize=14)
plt.savefig('heatmap_top50_DEGs.png', dpi=150, bbox_inches='tight')
plt.show()
print("Heatmap saved to heatmap_top50_DEGs.png")

### 7d. PCA of Normalized Counts


In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Use top 500 most variable genes
top_var_genes = norm_counts_df.var(axis=0).nlargest(500).index
pca_input     = norm_counts_df[top_var_genes]

scaler = StandardScaler()
pca    = PCA(n_components=2)
pca_coords = pca.fit_transform(scaler.fit_transform(pca_input))

fig, ax = plt.subplots(figsize=(7, 6))
for cond, grp_idx in metadata.groupby('condition').groups.items():
    mask = [s in grp_idx for s in norm_counts_df.index]
    ax.scatter(pca_coords[mask, 0], pca_coords[mask, 1],
               label=cond, color=palette[cond], s=100, edgecolors='k')

for i, sample in enumerate(norm_counts_df.index):
    ax.annotate(sample, (pca_coords[i, 0], pca_coords[i, 1]),
                fontsize=8, ha='center', va='bottom',
                xytext=(0, 6), textcoords='offset points')

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)', fontsize=12)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=12)
ax.set_title('PCA of Normalized Counts (top 500 variable genes)', fontsize=13)
ax.legend()
ax.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig('PCA_normalized_counts.png', dpi=150, bbox_inches='tight')
plt.show()
print("PCA plot saved to PCA_normalized_counts.png")

### 7e. Bar Chart – Up vs Down Regulated Gene Counts


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
categories_bar = ['Up-regulated', 'Down-regulated']
counts_bar     = [len(deg_up), len(deg_down)]
bar_colors_bar = ['#D95F02', '#1B9E77']

bars = ax.bar(categories_bar, counts_bar, color=bar_colors_bar, width=0.5)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + max(counts_bar) * 0.02,
            str(int(bar.get_height())),
            ha='center', va='bottom', fontweight='bold', fontsize=12)

ax.set_ylabel('Number of Genes', fontsize=12)
ax.set_title(f'DEGs (padj < {PADJ_THRESHOLD}, |logFC| ≥ {LFC_THRESHOLD})\nsbp-1 RNAi vs Control',
             fontsize=12)
ax.set_ylim(0, max(counts_bar) * 1.15)
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('bar_chart_DEG_counts.png', dpi=150, bbox_inches='tight')
plt.show()
print("Bar chart saved to bar_chart_DEG_counts.png")

## 8. Export Results


In [ ]:
# All results (sorted by adjusted p-value)
results.to_csv('GSE70692_sbp1RNAi_DESeq2_all_genes.tsv', sep='\t')

# Significant DEG subsets
deg_up.to_csv('GSE70692_sbp1RNAi_DESeq2_DEGs_up.tsv',       sep='\t')
deg_down.to_csv('GSE70692_sbp1RNAi_DESeq2_DEGs_down.tsv',   sep='\t')
deg_all.to_csv('GSE70692_sbp1RNAi_DESeq2_DEGs_P0.05_FC2.tsv', sep='\t')

print("Results exported:")
print(f"  All genes:          GSE70692_sbp1RNAi_DESeq2_all_genes.tsv          ({len(results)} rows)")
print(f"  Up-regulated DEGs:  GSE70692_sbp1RNAi_DESeq2_DEGs_up.tsv           ({len(deg_up)} rows)")
print(f"  Down-regulated DEGs:GSE70692_sbp1RNAi_DESeq2_DEGs_down.tsv         ({len(deg_down)} rows)")
print(f"  DEGs (|logFC|>=1):  GSE70692_sbp1RNAi_DESeq2_DEGs_P0.05_FC2.tsv   ({len(deg_all)} rows)")

## Summary

| Step | Method | Output |
|------|--------|--------|
| Normalization | DESeq2 median-of-ratios | Size factors, normalized count matrix |
| Differential expression | Negative binomial GLM + Wald test | log2 fold change, p-value, adjusted p-value |
| Multiple testing correction | Benjamini-Hochberg FDR | `padj` column |
| Thresholds | padj < 0.05 AND \|log2FC\| ≥ 1 | Up/down DEG lists |
| Visualizations | Volcano, MA, Heatmap, PCA, Bar chart | PNG files |

### Key References
- Love MI, Huber W, Anders S (2014). *Moderated estimation of fold change and
  dispersion for RNA-seq data with DESeq2*. Genome Biology, 15:550.
- Muzellec B, Telenczuk M, Cabeli V, Andreux M (2023). *PyDESeq2: a python
  package for bulk RNA-seq differential expression analysis*. Bioinformatics.
- GSE70692: sbp-1 RNAi C. elegans RNA-seq dataset. NCBI GEO.
